# Steam Price Modeling

This notebook trains baseline models for market-aligned Steam game list-price prediction. The target is `log_list_price`, and the first model family uses pre-release metadata only.

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "games_price_model.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DATA_PATH = DATA_DIR / "games_price_model.csv"

## 1. Load modeling data

The feature-engineering notebook already filters to valid paid games with estimated list prices between `$0.49` and `$99.99`.

In [3]:
def parse_list(value):
    if pd.isna(value):
        return []
    return ast.literal_eval(value)


df = pd.read_csv(MODEL_DATA_PATH)
for col in ["genre_list", "tag_list", "category_list"]:
    df[col] = df[col].apply(parse_list)

print(f"Shape: {df.shape}")
print("Valid price targets:", df["valid_price_target"].sum())
print("Estimated list price range:", df["estimated_list_price"].min(), df["estimated_list_price"].max())
df.head()

Shape: (96062, 42)
Valid price targets: 96062
Estimated list price range: 0.49 99.99


,AppID,Name,current_price,Discount,has_active_discount,estimated_list_price,log_list_price,price_target_source,valid_price_target,Release date,Release date parsed,Estimated owners,Genres,Tags,Categories,Developers,Publishers,Windows,Mac,Linux,required_age_clipped,platform_count,Release year,release_age_years,dlc_count_clipped,achievements_clipped,log1p_dlc_count,log1p_achievements,genre_count,tag_count,category_count,genre_list,tag_list,category_list,review_count,positive_share,log1p_review_count,log1p_recommendations,log1p_peak_ccu,log1p_avg_playtime_forever,log1p_median_playtime_forever,Metacritic score
0,496350,Supipara - Chapter 1 Spring Has Come!,5.24,65,True,14.97,2.770712,reconstructed,True,"Jul 29, 2016",2016-07-29,0 - 20000,Adventure,"Adventure,Visual Novel,Anime,Cute","Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",minori,MangaGamer,True,False,False,0,1,2016,10.0,0,0,0.000000,0.000000,1,4,4,[Adventure],"[Adventure, Visual Novel, Anime, Cute]","[Single-player, Steam Trading Cards, Steam Cloud, Family Sharing]",255,0.988235,5.545177,5.446737,0.000000,2.197225,2.197225,0
1,1034400,Mystery Solitaire The Black Raven,4.99,0,False,4.99,1.790091,observed,True,"May 6, 2019",2019-05-06,0 - 20000,Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,Mystery,Atmospheric,Family Friendly,PvE,Tutorial,Singleplayer...","Single-player,Family Sharing",Somer Games,8floor,True,True,False,0,2,2019,7.0,0,0,0.000000,0.000000,1,16,2,[Casual],"[Casual, Card Game, Solitaire, Puzzle, Hidden Object, 2D, Colorful, Stylized, Logic, Mystery, Atmospheric, Family Friendly, PvE, Tutoria...","[Single-player, Family Sharing]",24,0.875000,3.218876,0.000000,0.000000,0.000000,0.000000,0
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,False,8.99,2.301585,observed,True,"Oct 31, 2024",2024-10-31,0 - 20000,"Casual,Indie,Simulation",NaN,"Single-player,Steam Achievements,Family Sharing",유진게임즈,유진게임즈,True,False,False,0,1,2024,2.0,1,19,0.693147,2.995732,3,0,3,"[Casual, Indie, Simulation]",[],"[Single-player, Steam Achievements, Family Sharing]",0,NaN,0.000000,0.000000,0.693147,0.000000,0.000000,0
3,3631080,Maze Quest VR,4.99,0,False,4.99,1.790091,observed,True,"Apr 24, 2025",2025-04-24,0 - 20000,"Action,Early Access",NaN,"Single-player,VR Only,Steam Leaderboards,Family Sharing",Reality Expanded LLC,Reality Expanded LLC,True,False,False,0,1,2025,1.0,0,0,0.000000,0.000000,2,0,4,"[Action, Early Access]",[],"[Single-player, VR Only, Steam Leaderboards, Family Sharing]",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,1654170,Agony VR,13.99,0,False,13.99,2.707383,observed,True,"Apr 5, 2023",2023-04-05,0 - 20000,"Action,Adventure",NaN,"Single-player,Tracked Controller Support,VR Only,Family Sharing",Ignibit,"Ignibit,Madmind Studio",True,False,False,0,1,2023,3.0,0,0,0.000000,0.000000,2,0,4,"[Action, Adventure]",[],"[Single-player, Tracked Controller Support, VR Only, Family Sharing]",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [4]:
df["estimated_list_price"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("estimated_list_price")

,estimated_list_price
count,96062.000000
mean,7.957556
std,8.848327
min,0.490000
1%,0.490000
5%,0.990000
10%,0.990000
25%,2.390000
50%,4.990000
75%,9.990000


## 2. Feature sets

This first pass uses pre-release features only. Outcome variables such as reviews, recommendations, CCU, and playtime are held back for a later gamer-facing value model.

In [5]:
TARGET = "log_list_price"

ID_COLS = ["AppID", "Name"]
NUMERIC_FEATURES = [
    "required_age_clipped",
    "platform_count",
    "Release year",
    "release_age_years",
    "dlc_count_clipped",
    "achievements_clipped",
    "log1p_dlc_count",
    "log1p_achievements",
    "genre_count",
    "tag_count",
    "category_count",
]
BOOLEAN_FEATURES = ["Windows", "Mac", "Linux"]
MULTILABEL_FEATURES = ["genre_list", "tag_list", "category_list"]

TAG_MIN_COUNT = 100
CATEGORY_MIN_COUNT = 100


def label_counts(series):
    return series.explode().loc[lambda s: s.notna() & s.ne("")].value_counts()


genre_vocab = sorted(label_counts(df["genre_list"]).index.tolist())
tag_vocab = sorted(label_counts(df["tag_list"]).loc[lambda s: s >= TAG_MIN_COUNT].index.tolist())
category_vocab = sorted(label_counts(df["category_list"]).loc[lambda s: s >= CATEGORY_MIN_COUNT].index.tolist())

feature_summary = pd.Series({
    "rows": len(df),
    "numeric_features": len(NUMERIC_FEATURES),
    "boolean_features": len(BOOLEAN_FEATURES),
    "genre_features": len(genre_vocab),
    "tag_features": len(tag_vocab),
    "category_features": len(category_vocab),
})
feature_summary.to_frame("count")

,count
rows,96062
numeric_features,11
boolean_features,3
genre_features,33
tag_features,390
category_features,49


## 3. Train/test split

The split is random for the baseline. Later, we can test time-aware validation if release year proves important.

In [6]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

y_train = train_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "median_price": [train_df["estimated_list_price"].median(), test_df["estimated_list_price"].median()],
    "mean_price": [train_df["estimated_list_price"].mean(), test_df["estimated_list_price"].mean()],
    "reconstructed_pct": [
        train_df["price_target_source"].eq("reconstructed").mean() * 100,
        test_df["price_target_source"].eq("reconstructed").mean() * 100,
    ],
})
split_summary.round(2)

,split,rows,median_price,mean_price,reconstructed_pct
0,train,76849,4.99,7.94,42.2
1,test,19213,4.99,8.02,42.5


## 4. Build sparse feature matrix

Ridge and KNN both use the same pre-release feature matrix. Numeric features are imputed and standardized; multi-label features are multi-hot encoded.

In [7]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("boolean", SimpleImputer(strategy="most_frequent"), BOOLEAN_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

genre_binarizer = MultiLabelBinarizer(classes=genre_vocab, sparse_output=True)
tag_binarizer = MultiLabelBinarizer(classes=tag_vocab, sparse_output=True)
category_binarizer = MultiLabelBinarizer(classes=category_vocab, sparse_output=True)


def keep_vocab(items, vocab):
    vocab = set(vocab)
    return [item for item in items if item in vocab]


def build_feature_matrix(frame, fit=False):
    frame_for_transform = frame.copy()
    frame_for_transform[BOOLEAN_FEATURES] = frame_for_transform[BOOLEAN_FEATURES].astype(float)
    frame_for_transform["genre_list"] = frame_for_transform["genre_list"].apply(lambda items: keep_vocab(items, genre_vocab))
    frame_for_transform["tag_list"] = frame_for_transform["tag_list"].apply(lambda items: keep_vocab(items, tag_vocab))
    frame_for_transform["category_list"] = frame_for_transform["category_list"].apply(lambda items: keep_vocab(items, category_vocab))

    if fit:
        numeric_matrix = numeric_transformer.fit_transform(frame_for_transform)
        genre_matrix = genre_binarizer.fit_transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.fit_transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.fit_transform(frame_for_transform["category_list"])
    else:
        numeric_matrix = numeric_transformer.transform(frame_for_transform)
        genre_matrix = genre_binarizer.transform(frame_for_transform["genre_list"])
        tag_matrix = tag_binarizer.transform(frame_for_transform["tag_list"])
        category_matrix = category_binarizer.transform(frame_for_transform["category_list"])

    numeric_matrix = sparse.csr_matrix(numeric_matrix)
    return sparse.hstack([numeric_matrix, genre_matrix, tag_matrix, category_matrix], format="csr")


X_train = build_feature_matrix(train_df, fit=True)
X_test = build_feature_matrix(test_df, fit=False)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Sparsity:", 1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]))

X_train: (76849, 486)
X_test: (19213, 486)
Sparsity: 0.9365216604825014


In [8]:
feature_names = (
    NUMERIC_FEATURES
    + BOOLEAN_FEATURES
    + [f"genre={name}" for name in genre_binarizer.classes_]
    + [f"tag={name}" for name in tag_binarizer.classes_]
    + [f"category={name}" for name in category_binarizer.classes_]
)

print("Feature names:", len(feature_names))
feature_names[:20]

Feature names: 486


['required_age_clipped',
 'platform_count',
 'Release year',
 'release_age_years',
 'dlc_count_clipped',
 'achievements_clipped',
 'log1p_dlc_count',
 'log1p_achievements',
 'genre_count',
 'tag_count',
 'category_count',
 'Windows',
 'Mac',
 'Linux',
 'genre=360 Video',
 'genre=Accounting',
 'genre=Action',
 'genre=Adventure',
 'genre=Animation & Modeling',
 'genre=Audio Production']

## 5. Evaluation helpers

Models train on log price, but evaluation is reported in dollars for business readability.

In [9]:
def to_price(log_values):
    return np.expm1(log_values).clip(min=0.49)


def evaluate_predictions(model_name, y_true_log, y_pred_log):
    actual = to_price(y_true_log)
    predicted = to_price(y_pred_log)
    errors = np.abs(actual - predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5

    return {
        "model": model_name,
        "mae_dollars": mean_absolute_error(actual, predicted),
        "median_ae_dollars": median_absolute_error(actual, predicted),
        "rmse_dollars": rmse,
        "mean_actual_price": actual.mean(),
        "mean_predicted_price": predicted.mean(),
        "pct_within_25pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.25).mean() * 100,
        "pct_within_50pct": (np.maximum(actual, predicted) / np.minimum(actual, predicted) <= 1.50).mean() * 100,
    }


def price_band_table(frame, y_pred_log):
    result = frame[["estimated_list_price"]].copy()
    result["predicted_price"] = to_price(y_pred_log)
    result["abs_error"] = (result["estimated_list_price"] - result["predicted_price"]).abs()
    result["price_band"] = pd.cut(
        result["estimated_list_price"],
        bins=[0, 2.99, 4.99, 9.99, 19.99, 49.99, 100],
        labels=["0.49-2.99", "3-4.99", "5-9.99", "10-19.99", "20-49.99", "50-99.99"],
        include_lowest=True,
    )
    return (
        result.groupby("price_band", observed=False)
        .agg(
            games=("estimated_list_price", "size"),
            median_actual=("estimated_list_price", "median"),
            median_predicted=("predicted_price", "median"),
            mae=("abs_error", "mean"),
            median_ae=("abs_error", "median"),
        )
    )

## 6. Median baseline

This baseline predicts the same median log price for every game. Every real model should beat it.

In [10]:
median_model = DummyRegressor(strategy="median")
median_model.fit(X_train, y_train)
median_pred_log = median_model.predict(X_test)

results = [evaluate_predictions("Median baseline", y_test, median_pred_log)]
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.03,9.382,8.017,4.99,17.608,28.309


## 7. Ridge regression baseline

Ridge is a strong first model for sparse multi-hot metadata. It is fast, stable, and interpretable enough for baseline analysis.

In [11]:
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train, y_train)
ridge_pred_log = ridge_model.predict(X_test)

results.append(evaluate_predictions("Ridge", y_test, ridge_pred_log))
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408


In [12]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef_log_price": ridge_model.coef_,
})

display(coef_df.sort_values("coef_log_price", ascending=False).head(20))
display(coef_df.sort_values("coef_log_price").head(20))

,feature,coef_log_price
44,genre=Video Production,0.332364
27,genre=Game Development,0.312831
483,category=VR Only,0.254412
9,tag_count,0.250576
287,tag=Otome,0.246554
23,genre=Early Access,0.245162
485,category=VR Supported,0.235017
358,tag=Software Training,0.199207
403,tag=Trains,0.199018
447,category=Full controller support,0.194383


,feature,coef_log_price
446,category=Family Sharing,-0.554068
43,genre=Utilities,-0.397698
319,tag=RPGMaker,-0.247686
169,tag=Experience,-0.243095
350,tag=Short,-0.234887
117,tag=Clicker,-0.232201
257,tag=Minimalist,-0.218116
33,genre=Photo Editing,-0.214228
473,category=Steam Achievements,-0.210877
252,tag=Memes,-0.206392


## 8. KNN median-price baseline

This baseline predicts each test game's price as the median list price of its nearest training neighbors. It is slower than Ridge, but useful because it resembles the future comparable-games layer.

In [13]:
K_NEIGHBORS = 25

knn_model = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", algorithm="brute", n_jobs=-1)
knn_model.fit(X_train)

neighbor_distances, neighbor_indices = knn_model.kneighbors(X_test)
neighbor_prices = train_df.iloc[neighbor_indices.ravel()]["estimated_list_price"].to_numpy().reshape(neighbor_indices.shape)
knn_pred_price = np.median(neighbor_prices, axis=1)
knn_pred_log = np.log1p(knn_pred_price)

results.append(evaluate_predictions(f"KNN median k={K_NEIGHBORS}", y_test, knn_pred_log))
pd.DataFrame(results).round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408
2,KNN median k=25,4.311,2.750,7.406,8.017,6.354,22.979,37.292


## 9. Model comparison and error by price band

In [14]:
comparison = pd.DataFrame(results).sort_values("mae_dollars")
comparison.round(3)

,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
2,KNN median k=25,4.311,2.750,7.406,8.017,6.354,22.979,37.292
1,Ridge,4.510,2.707,7.688,8.017,6.182,20.189,36.408
0,Median baseline,5.393,3.030,9.382,8.017,4.990,17.608,28.309


In [15]:
display(price_band_table(test_df, median_pred_log).round(2).rename_axis("Median baseline"))
display(price_band_table(test_df, ridge_pred_log).round(2).rename_axis("Ridge"))
display(price_band_table(test_df, knn_pred_log).round(2).rename_axis("KNN median"))

,games,median_actual,median_predicted,mae,median_ae
Median baseline,,,,,
0.49-2.99,6567,1.97,4.99,3.22,3.02
3-4.99,3834,4.96,4.99,0.41,0.03
5-9.99,4641,7.99,4.99,3.29,3.00
10-19.99,3123,14.99,4.99,10.71,10.00
20-49.99,921,29.98,4.99,26.92,24.99
50-99.99,127,59.98,4.99,58.07,54.99


,games,median_actual,median_predicted,mae,median_ae
Ridge,,,,,
0.49-2.99,6567,1.97,3.95,2.66,2.33
3-4.99,3834,4.96,4.62,1.71,1.22
5-9.99,4641,7.99,5.44,3.24,2.95
10-19.99,3123,14.99,7.41,7.71,7.54
20-49.99,921,29.98,11.13,19.18,18.61
50-99.99,127,59.98,12.32,46.12,48.92


,games,median_actual,median_predicted,mae,median_ae
KNN median,,,,,
0.49-2.99,6567,1.97,3.98,2.50,2.00
3-4.99,3834,4.96,4.95,1.88,1.01
5-9.99,4641,7.99,4.99,3.32,3.00
10-19.99,3123,14.99,7.99,7.32,7.00
20-49.99,921,29.98,12.99,17.37,17.50
50-99.99,127,59.98,20.99,38.83,40.01


## 10. Price alignment labels

Use the best baseline model from this notebook to label each test game as below-market, market-aligned, or above-market.

In [16]:
best_pred_log = knn_pred_log
best_model_name = f"KNN median k={K_NEIGHBORS}"

predictions = test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
predictions["predicted_market_price"] = to_price(best_pred_log)
predictions["price_ratio"] = predictions["estimated_list_price"] / predictions["predicted_market_price"]

predictions["price_alignment"] = pd.cut(
    predictions["price_ratio"],
    bins=[0, 0.75, 1.25, np.inf],
    labels=["below-market", "market-aligned", "above-market"],
)

predictions["abs_error"] = (predictions["estimated_list_price"] - predictions["predicted_market_price"]).abs()

alignment_summary = (
    predictions.groupby("price_alignment", observed=False)
    .agg(
        games=("AppID", "size"),
        median_actual=("estimated_list_price", "median"),
        median_predicted=("predicted_market_price", "median"),
        median_ratio=("price_ratio", "median"),
    )
)
alignment_summary.round(2)

,games,median_actual,median_predicted,median_ratio
price_alignment,,,,
below-market,6571,1.99,4.99,0.41
market-aligned,4879,4.99,4.99,1.00
above-market,7763,9.99,4.98,2.00


## 11. Example inspections

These examples are not final product claims. They help identify whether the model's errors and labels look reasonable.

In [17]:
EXAMPLE_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio", "price_alignment",
    "Discount", "price_target_source", "Genres", "Tags", "review_count", "log1p_recommendations",
]

In [18]:
POPULAR_EXAMPLE_COLS = [
    "Name", "estimated_list_price", "predicted_market_price", "price_ratio", "price_alignment",
    "Discount", "price_target_source", "Genres", "review_count", "log1p_recommendations",
]

predictions.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, POPULAR_EXAMPLE_COLS].head(20)


,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,review_count,log1p_recommendations
8620,Terraria,9.98,14.95,0.667559,below-market,50,reconstructed,"Action,Adventure,Indie,RPG",1409473,13.966525
34598,Black Myth: Wukong,59.99,49.90,1.202204,market-aligned,0,observed,"Action,Adventure,RPG",1150098,13.668195
86902,Left 4 Dead 2,1.99,9.95,0.200000,below-market,0,observed,Action,963983,13.555354
93023,Lethal Company,9.99,6.98,1.431232,above-market,25,reconstructed,"Action,Adventure,Indie,Early Access",487160,12.905575
42703,DayZ,49.98,27.99,1.785638,above-market,55,reconstructed,"Action,Adventure,Massively Multiplayer",430252,12.822468
21270,Don't Starve Together,14.97,24.98,0.599279,below-market,66,reconstructed,"Action,Adventure,Indie,RPG,Simulation,Strategy",499061,12.791507
39317,Monster Hunter: World,29.97,39.95,0.750188,market-aligned,67,reconstructed,Action,493390,12.642678
19496,Raft,19.99,19.98,1.000501,market-aligned,33,reconstructed,"Adventure,Indie,Simulation",346359,12.635137
86671,Subnautica,29.96,14.97,2.001336,above-market,75,reconstructed,"Adventure,Indie",313707,12.612730
78298,Deep Rock Galactic,29.97,29.98,0.999666,market-aligned,70,reconstructed,Action,343000,12.578333


In [19]:
predictions.sort_values("price_ratio").loc[:, EXAMPLE_COLS].head(20)

,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,Tags,review_count,log1p_recommendations
93017,Cooperate,0.98,29.99,0.032678,below-market,45,reconstructed,"Casual,RPG","RPG,Casual,Action RPG,Party-Based RPG,3D Platformer,Mystery Dungeon,2D,1980s,Story Rich,Singleplayer",1,0.000000
423,Mitigate,0.99,29.99,0.033011,below-market,0,observed,"Casual,RPG","RPG,Casual,Action-Adventure,Animation & Modeling,Traditional Roguelike,Pixel Graphics,Anime,Transportation,Singleplayer",1,0.000000
59958,Molest,0.89,24.99,0.035614,below-market,0,observed,"Casual,RPG","Casual,Adventure,4X,Farming Sim,Roguevania,Mystery Dungeon,2D,360 Video,LGBTQ+,RPG,Medieval,Dynamic Narration,Singleplayer",2,0.000000
54876,Barro Racing,0.49,12.47,0.039294,below-market,0,observed,"Casual,Indie,Racing","Indie,Racing,Sports,Multiplayer,PvP,Arcade,Driving,Automobile Sim,Immersive Sim,Simulation,Time Attack,Local Multiplayer,Singleplayer,Sp...",975,7.003974
40514,Ice Station Z,0.99,24.98,0.039632,below-market,0,observed,"Action,Adventure,Indie","Exploration,Third-Person Shooter,Sandbox,Perma Death,PvE,PvP,Action-Adventure,Character Customization,3D,Third Person,Controller,Open Wo...",177,5.129899
32030,Trump VS Covid: Save The World Clicker,0.49,10.97,0.044667,below-market,0,observed,"Indie,Simulation,Strategy","Clicker,Simulation,Idler,Outbreak Sim,Strategy,Political Sim,Management,Politics,America,Capitalism,Economy,Dark Humor,Experimental,Gran...",35,0.000000
24506,Chromosome Evil,1.34,29.97,0.044711,below-market,0,observed,Strategy,"War,Atmospheric,Singleplayer,Strategy,Simulation,Choices Matter,Resource Management,RTS,Post-apocalyptic,Zombies,Survival,Psychological,...",321,5.758902
23293,I Know This Place..? chapter I (prologue),0.49,9.99,0.049049,below-market,0,observed,"Adventure,Indie","Detective,1990's,Exploration,Atmospheric,Puzzle,Surreal,Visual Novel,Hidden Object,Sci-fi,3D,First-Person,Alternate History,Choose Your ...",138,4.882802
73222,Resoraki: The racing,0.49,9.99,0.049049,below-market,0,observed,"Action,Adventure,Casual,Indie,Racing,Simulation,Sports",NaN,0,0.000000
37731,Dots in line,0.49,9.99,0.049049,below-market,0,observed,Casual,"Match 3,Score Attack,Relaxing,Puzzle,Casual,Singleplayer,2D,Tabletop,Solitaire,Trivia,Card Game,Hidden Object,Top-Down,2.5D,Colorful,Dec...",9,0.000000


In [20]:
predictions.sort_values("price_ratio", ascending=False).loc[:, EXAMPLE_COLS].head(20)

,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,Tags,review_count,log1p_recommendations
76564,GlobalMap Astro,54.99,1.00,54.990000,above-market,0,observed,"Casual,Indie,Education","Education,Indie,Casual,Space",10,0.000000
33735,Canyon of Outlaws,39.80,1.00,39.800000,above-market,95,reconstructed,"Adventure,Casual,Racing,Sports","Adventure,Casual,Racing,Sports,Bullet Hell,Card Battler,Collectathon,Singleplayer",103,0.000000
49536,FORAY,69.99,1.99,35.170854,above-market,0,observed,Casual,NaN,0,0.000000
55624,Forsaken Manor,99.99,2.99,33.441472,above-market,0,observed,"Casual,Indie",NaN,0,0.000000
68150,Princess Maker ~Faery Tales Come True~ (HD Remake),29.98,0.99,30.282828,above-market,50,reconstructed,Simulation,"Simulation,Female Protagonist,Multiple Endings,Singleplayer,Anime,Classic",230,5.455321
54562,Dumb Witchfinder,54.99,1.99,27.633166,above-market,0,observed,"Casual,Indie","Side Scroller,Casual,2D Platformer,PvE,Arcade,Indie,Rogue-like,2D,Third Person,Pixel Graphics,Atmospheric,Combat,Singleplayer",2,0.000000
68066,Shiny Summer,49.80,1.96,25.408163,above-market,95,reconstructed,"Adventure,Casual,Indie","Anime,Puzzle,Casual,Cute,Relaxing,Colorful,Adventure,Singleplayer,2D,Indie,Clicker,JRPG,Simulation,Strategy",41,0.000000
87751,Nekomimi Nikki,49.80,1.96,25.408163,above-market,95,reconstructed,"Action,Adventure,Indie","Indie,Adventure,Action,Casual,Puzzle,2D,Anime,Singleplayer,Colorful,Cute,Atmospheric,Mature,Sexual Content,Story Rich,Visual Novel,Datin...",19,0.000000
42560,DORAEMON STORY OF SEASONS,49.97,1.98,25.237374,above-market,60,reconstructed,"Casual,Simulation","Farming Sim,Life Sim,RPG,Family Friendly,Adventure,Simulation,Singleplayer,Cute,2D,Hand-drawn,Relaxing,Colorful,Isometric,Atmospheric,Cr...",2143,7.510978
54498,Bright Bob,24.90,0.99,25.151515,above-market,90,reconstructed,"Casual,Indie","Indie,Casual,Fast-Paced,2D,Atmospheric,Singleplayer,Music,Side Scroller",49,0.000000


In [21]:
predictions.sort_values("abs_error", ascending=False).loc[:, EXAMPLE_COLS + ["abs_error"]].head(20)

,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,Tags,review_count,log1p_recommendations,abs_error
55624,Forsaken Manor,99.99,2.99,33.441472,above-market,0,observed,"Casual,Indie",NaN,0,0.000000,97.00
34138,Mocap Fusion [ VR ],99.99,4.96,20.159274,above-market,0,observed,"Casual,Simulation","Simulation,Animation & Modeling,Education,Video Production,Choose Your Own Adventure,Exploration,3D,Cinematic,Realistic,Building,Game De...",18,0.000000,95.03
84253,Call of Duty®: Ghosts - Digital Hardened Edition,99.98,6.98,14.323782,above-market,60,reconstructed,Action,"Action,FPS,Dog,Multiplayer,Shooter,Singleplayer,Casual,First-Person,Family Friendly",36,0.000000,93.00
7217,ACTION GAME MAKER,99.99,7.99,12.514393,above-market,20,reconstructed,"Animation & Modeling,Design & Illustration,Software Training,Web Publishing,Game Development",NaN,0,0.000000,92.00
67801,VideoPad Video Editor,99.98,9.99,10.008008,above-market,35,reconstructed,Video Production,"Video Production,Utilities,Software",71,0.000000,89.99
12950,CyberLink PowerDirector 21 Ultimate,99.99,10.99,9.098271,above-market,0,observed,Video Production,Video Production,10,0.000000,89.00
47902,Samplitude Music Studio 2019 Steam Edition,99.00,14.98,6.608812,above-market,80,reconstructed,Audio Production,Audio Production,22,0.000000,84.02
7978,Franchise Hockey Manager 4,79.96,8.95,8.934078,above-market,75,reconstructed,"Indie,Simulation,Sports,Strategy","Sports,Strategy,Simulation,Indie,Hockey",145,4.852030,71.01
78445,Quicken® WillMaker® Plus 2019 and Living Trust,79.98,8.99,8.896552,above-market,40,reconstructed,Accounting,"Psychological Horror,Perma Death,Souls-like,Nudity,Sexual Content,Anime,Hentai,Survival Horror,Gore,Horror",2,0.000000,70.99
49536,FORAY,69.99,1.99,35.170854,above-market,0,observed,Casual,NaN,0,0.000000,68.00


In [22]:
def search_predictions(query):
    mask = predictions["Name"].fillna("").str.contains(query, case=False, regex=False)
    return predictions.loc[mask, EXAMPLE_COLS].sort_values("review_count", ascending=False)


search_predictions("hades")

,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,Tags,review_count,log1p_recommendations
16413,Hades II,29.99,16.93,1.771412,above-market,25,reconstructed,"Action,Indie,RPG","Action,Rogue-like,Rogue-lite,Hack and Slash,RPG,Action Roguelike,Mythology,Dungeon Crawler,Action RPG,Isometric,Hand-drawn,Female Protag...",64132,11.510021
77137,Shades of Sakura,1.97,3.98,0.494975,below-market,61,reconstructed,"Adventure,Casual,Indie,RPG","Hentai,Dating Sim,Mature,Puzzle,Visual Novel,NSFW,Casual,Anime,Indie,Adventure,Singleplayer,Sexual Content,Story Rich,RPG,Female Protago...",556,6.269096
72560,The Road to Hades,4.95,4.99,0.991984,market-aligned,80,reconstructed,"Violent,Gore,Adventure,Indie","Gore,Adventure,Indie,Violent,Horror,Psychological Horror,VR",51,0.000000
85737,Lumins and Shades,14.99,2.97,5.047138,above-market,0,observed,"Casual,Indie,Strategy","Puzzle,Logic,Nonlinear,Exploration,Cute,Casual,Family Friendly,Pixel Graphics,2D,Strategy,Top-Down,Grid-Based Movement,Singleplayer,Atmo...",10,0.000000
47521,Cinders Of Hades,2.99,2.99,1.000000,market-aligned,0,observed,"Action,Casual","Bullet Hell,Shoot 'Em Up,Casual,Shooter,Arcade,3D,First-Person,VR,Dragons,Relaxing,Action,Demons,Fantasy,Magic,Physics,Destruction,Singl...",3,0.000000
83751,Grim Tales: All Shades of Black Collector's Edition,13.98,13.98,1.000000,market-aligned,50,reconstructed,"Adventure,Casual","Adventure,Casual,Hidden Object,Point & Click,Puzzle,Fantasy,Atmospheric,Singleplayer,2D",3,0.000000


## 12. Initial modeling takeaways

The full valid-price model is useful as a first baseline: both Ridge and KNN beat the median-price baseline. Inspection also shows a problem: the broad Steam dataset includes many zero-traction or very low-traction games, which may dilute the market signal for games that players and developers actually compare against.

Rather than silently changing the original setup, the next section tests stricter market scopes based on review/recommendation evidence.


## 13. Market-scope sensitivity

These scopes test whether removing near-zero-traction games improves price modeling. The filters are based on observable market evidence, not subjective title quality.


In [23]:
scope_masks = {
    "full_valid": pd.Series(True, index=df.index),
    "review_count >= 10": df["review_count"] >= 10,
    "review_count >= 50": df["review_count"] >= 50,
    "recommendations > 0": df["log1p_recommendations"] > 0,
}

scope_rows = []
for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    scope_rows.append({
        "scope": scope_name,
        "rows": len(scope_df),
        "pct_of_full": len(scope_df) / len(df) * 100,
        "median_price": scope_df["estimated_list_price"].median(),
        "mean_price": scope_df["estimated_list_price"].mean(),
        "median_review_count": scope_df["review_count"].median(),
        "median_recommendations_log": scope_df["log1p_recommendations"].median(),
        "reconstructed_pct": scope_df["price_target_source"].eq("reconstructed").mean() * 100,
    })

scope_summary_df = pd.DataFrame(scope_rows)
scope_summary_df.round(2)


,scope,rows,pct_of_full,median_price,mean_price,median_review_count,median_recommendations_log,reconstructed_pct
0,full_valid,96062,100.00,4.99,7.96,10.0,0.00,42.26
1,review_count >= 10,49139,51.15,5.99,9.48,60.0,0.00,53.65
2,review_count >= 50,26542,27.63,9.90,11.83,239.0,5.29,63.46
3,recommendations > 0,20211,21.04,9.99,13.55,403.0,6.02,68.24


In [24]:
def train_ridge_for_scope(scope_df, alpha=10.0, random_state=42):
    train_scope_df, test_scope_df = train_test_split(scope_df, test_size=0.20, random_state=random_state)
    y_scope_train = train_scope_df[TARGET].to_numpy()
    y_scope_test = test_scope_df[TARGET].to_numpy()

    X_scope_train = build_feature_matrix(train_scope_df, fit=True)
    X_scope_test = build_feature_matrix(test_scope_df, fit=False)

    model = Ridge(alpha=alpha, random_state=random_state)
    model.fit(X_scope_train, y_scope_train)
    pred_log = model.predict(X_scope_test)

    metrics = evaluate_predictions("Ridge", y_scope_test, pred_log)
    metrics.update({
        "train_rows": len(train_scope_df),
        "test_rows": len(test_scope_df),
        "test_median_price": test_scope_df["estimated_list_price"].median(),
        "test_mean_price": test_scope_df["estimated_list_price"].mean(),
    })
    return metrics, model, train_scope_df, test_scope_df, X_scope_train, X_scope_test, pred_log


In [25]:
scope_models = {}
scope_metrics = []

for scope_name, mask in scope_masks.items():
    scope_df = df.loc[mask].copy()
    metrics, model, scope_train_df, scope_test_df, X_scope_train, X_scope_test, pred_log = train_ridge_for_scope(scope_df)
    metrics["scope"] = scope_name
    scope_metrics.append(metrics)
    scope_models[scope_name] = {
        "model": model,
        "train_df": scope_train_df,
        "test_df": scope_test_df,
        "X_train": X_scope_train,
        "X_test": X_scope_test,
        "pred_log": pred_log,
    }

scope_ridge_results = pd.DataFrame(scope_metrics)
scope_ridge_results = scope_ridge_results[[
    "scope", "train_rows", "test_rows", "mae_dollars", "median_ae_dollars", "rmse_dollars",
    "pct_within_25pct", "pct_within_50pct", "test_median_price", "test_mean_price"
]].sort_values("mae_dollars")
scope_ridge_results.round(3)


,scope,train_rows,test_rows,mae_dollars,median_ae_dollars,rmse_dollars,pct_within_25pct,pct_within_50pct,test_median_price,test_mean_price
0,full_valid,76849,19213,4.510,2.707,7.688,20.189,36.408,4.99,8.017
1,review_count >= 10,39311,9828,4.911,3.104,7.879,21.530,38.472,5.99,9.405
2,review_count >= 50,21233,5309,5.693,3.743,8.639,24.053,41.571,9.90,11.724
3,recommendations > 0,16168,4043,6.160,4.078,9.090,26.193,44.200,9.98,13.392


## 14. KNN on selected market scope

If a filtered scope improves Ridge performance, run the KNN median-price baseline on that scope as a comparable-games-style estimator. The default selected scope is `review_count >= 10`, which removes completely unvalidated games while keeping enough market breadth.


In [26]:
SELECTED_SCOPE = "review_count >= 10"

selected = scope_models[SELECTED_SCOPE]
selected_train_df = selected["train_df"]
selected_test_df = selected["test_df"]
X_selected_train = selected["X_train"]
X_selected_test = selected["X_test"]
y_selected_test = selected_test_df[TARGET].to_numpy()

selected_knn = NearestNeighbors(n_neighbors=K_NEIGHBORS, metric="cosine", algorithm="brute", n_jobs=-1)
selected_knn.fit(X_selected_train)
selected_distances, selected_indices = selected_knn.kneighbors(X_selected_test)
selected_neighbor_prices = selected_train_df.iloc[selected_indices.ravel()]["estimated_list_price"].to_numpy().reshape(selected_indices.shape)
selected_knn_pred_price = np.median(selected_neighbor_prices, axis=1)
selected_knn_pred_log = np.log1p(selected_knn_pred_price)

selected_results = [
    evaluate_predictions(f"Ridge ({SELECTED_SCOPE})", y_selected_test, selected["pred_log"]),
    evaluate_predictions(f"KNN median k={K_NEIGHBORS} ({SELECTED_SCOPE})", y_selected_test, selected_knn_pred_log),
]
pd.DataFrame(selected_results).round(3)


,model,mae_dollars,median_ae_dollars,rmse_dollars,mean_actual_price,mean_predicted_price,pct_within_25pct,pct_within_50pct
0,Ridge (review_count >= 10),4.911,3.104,7.879,9.405,7.594,21.530,38.472
1,KNN median k=25 (review_count >= 10),4.813,3.000,7.742,9.405,7.977,25.275,39.662


In [27]:
selected_predictions = selected_test_df[[
    "AppID", "Name", "estimated_list_price", "current_price", "Discount", "price_target_source",
    "Genres", "Tags", "Categories", "review_count", "log1p_recommendations"
]].copy()
selected_predictions["predicted_market_price"] = to_price(selected_knn_pred_log)
selected_predictions["price_ratio"] = selected_predictions["estimated_list_price"] / selected_predictions["predicted_market_price"]
selected_predictions["price_alignment"] = pd.cut(
    selected_predictions["price_ratio"],
    bins=[0, 0.75, 1.25, np.inf],
    labels=["below-market", "market-aligned", "above-market"],
)

selected_predictions.sort_values(
    ["log1p_recommendations", "review_count"], ascending=False
).loc[:, POPULAR_EXAMPLE_COLS].head(20)


,Name,estimated_list_price,predicted_market_price,price_ratio,price_alignment,Discount,price_target_source,Genres,review_count,log1p_recommendations
51155,HELLDIVERS™ 2,39.99,49.97,0.800280,market-aligned,20,reconstructed,Action,1017635,13.577903
36828,Dead by Daylight,19.97,29.96,0.666555,below-market,60,reconstructed,Action,797216,13.352402
11198,ARK: Survival Evolved,14.98,19.99,0.749375,below-market,34,reconstructed,"Action,Adventure,Indie,Massively Multiplayer,RPG",730170,13.248252
88551,The Forest,19.96,14.96,1.334225,above-market,77,reconstructed,"Action,Adventure,Indie,Simulation",627791,13.228018
30254,Fallout 4,19.97,29.93,0.667224,below-market,60,reconstructed,RPG,392287,12.529941
55137,Hades,24.97,15.97,1.563557,above-market,70,reconstructed,"Action,Indie,RPG",279741,12.513469
59090,Mount & Blade II: Bannerlord,49.98,29.93,1.669896,above-market,50,reconstructed,"Action,Indie,RPG,Simulation,Strategy",264391,12.287537
31110,Borderlands 2,19.96,19.96,1.000000,market-aligned,75,reconstructed,"Action,RPG",295218,12.272366
51110,Hunt: Showdown 1896,29.98,39.98,0.749875,below-market,55,reconstructed,Action,248220,12.222276
78644,ULTRAKILL,24.99,8.98,2.782851,above-market,30,reconstructed,"Action,Indie,Early Access",169751,12.064284


## 15. Updated modeling decision

Fill this in after reviewing the scope-sensitivity outputs:

- Whether the main model should stay on `full_valid` or move to `review_count >= 10`.
- Whether the filtered scope improves popular-title predictions enough to justify excluding zero-traction games.
- Whether KNN should remain a baseline estimator or move mostly into the comparable-games layer.
- Whether the next notebook should tune KNN `k`, add post-release features, or try boosting/dimensionality reduction.
